# SkyFusion YOLO26 Final Project

This notebook follows the final-project flow: install/import YOLO, add a custom block/backbone, create the custom YOLO26 YAML, train on SkyFusion at `imgsz=640`, then test `best.pt` on the SkyFusion test split.

Upload the SkyFusion dataset zip from Canvas or put it in Google Drive before running the dataset cell.

In [ ]:
# 1. Install dependencies
!pip -q install ultralytics pyyaml

import os
from pathlib import Path

import torch
import yaml
from ultralytics import YOLO

ROOT = Path('/content/skyfusion_final')
ROOT.mkdir(parents=True, exist_ok=True)
os.chdir(ROOT)

print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
print('device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## Dataset

Run either the Drive path option or the upload option. The code then unzips the dataset and normalizes `data.yaml` so Ultralytics can find `train`, `val`, and `test`.

In [ ]:
# 2. Load the SkyFusion dataset zip
# Option A: Google Drive. Put the Canvas zip in MyDrive and set this path.
USE_DRIVE = False
DRIVE_ZIP = '/content/drive/MyDrive/skyfusion.v1i.yolov11.zip'

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    dataset_zip = Path(DRIVE_ZIP)
else:
    from google.colab import files
    uploaded = files.upload()
    dataset_zip = Path(next(iter(uploaded.keys()))).resolve()

DATASET_DIR = ROOT / 'skyfusion_dataset'
DATASET_DIR.mkdir(parents=True, exist_ok=True)
!unzip -q -o "{dataset_zip}" -d "{DATASET_DIR}"

yaml_files = list(DATASET_DIR.rglob('data.yaml'))
assert yaml_files, 'No data.yaml found in the dataset zip.'
DATA_YAML = yaml_files[0]

with open(DATA_YAML, 'r') as f:
    data = yaml.safe_load(f)

dataset_root = DATA_YAML.parent
data['path'] = str(dataset_root)
data['train'] = str((dataset_root / 'train' / 'images').resolve())
data['val'] = str((dataset_root / 'valid' / 'images').resolve())
data['test'] = str((dataset_root / 'test' / 'images').resolve())
data['nc'] = 3
data['names'] = ['Aircraft', 'ship', 'vehicle']

COLAB_DATA_YAML = ROOT / 'skyfusion_data.yaml'
with open(COLAB_DATA_YAML, 'w') as f:
    yaml.safe_dump(data, f, sort_keys=False)

print(COLAB_DATA_YAML)
print(yaml.safe_dump(data, sort_keys=False))

## Custom Block and YOLO26 Model

This keeps the same architecture used in the project files: `SkyFusionBackbone` with `SkyFusionBlock`, P2/P3/P4/P5 detection, `end2end=True`, and `reg_max=16`.

In [ ]:
# 3. Define and register the custom SkyFusion block/backbone
from typing import Sequence

import torch.nn as nn
from ultralytics.nn.modules import Conv, DWConv
import ultralytics.nn.tasks as tasks


class DropPath(nn.Module):
    def __init__(self, drop_prob=0.0):
        super().__init__()
        self.drop_prob = float(drop_prob)

    def forward(self, x):
        if self.drop_prob == 0.0 or not self.training:
            return x
        keep_prob = 1 - self.drop_prob
        shape = (x.shape[0],) + (1,) * (x.ndim - 1)
        random_tensor = keep_prob + torch.rand(shape, dtype=x.dtype, device=x.device)
        random_tensor.floor_()
        return x.div(keep_prob) * random_tensor


class SkyFusionContextGate(nn.Module):
    def __init__(self, channels: int, reduction: int = 16, spatial_kernel: int = 7):
        super().__init__()
        hidden = max(channels // max(reduction, 1), 8)
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.channel_mlp = nn.Sequential(
            nn.Conv2d(channels, hidden, kernel_size=1, bias=True),
            nn.SiLU(inplace=True),
            nn.Conv2d(hidden, channels, kernel_size=1, bias=True),
        )
        self.spatial = nn.Conv2d(2, 1, kernel_size=spatial_kernel, padding=spatial_kernel // 2, bias=False)

    def forward(self, x):
        channel_gate = torch.sigmoid(self.channel_mlp(self.avg_pool(x)) + self.channel_mlp(self.max_pool(x)))
        x = x * channel_gate
        avg_map = torch.mean(x, dim=1, keepdim=True)
        max_map = torch.max(x, dim=1, keepdim=True)[0]
        spatial_gate = torch.sigmoid(self.spatial(torch.cat((avg_map, max_map), dim=1)))
        return x * spatial_gate


class SkyFusionBlock(nn.Module):
    def __init__(self, c1: int, c2: int, stride: int = 1, expansion: float = 2.0, drop_path: float = 0.0, layer_scale_init_value: float = 1e-4):
        super().__init__()
        hidden = max(int(c2 * expansion), c2)
        self.expand = Conv(c1, hidden, k=1, s=1)
        self.local_mixer = DWConv(hidden, hidden, k=5, s=stride)
        self.wide_mixer = DWConv(hidden, hidden, k=3, s=1, d=2)
        self.project = Conv(hidden, c2, k=1, s=1, act=False)
        self.gate = SkyFusionContextGate(c2)
        self.shortcut = nn.Identity() if c1 == c2 and stride == 1 else Conv(c1, c2, k=1, s=stride, act=False)
        self.drop_path = DropPath(drop_path)
        self.act = nn.SiLU(inplace=True)
        self.layer_scale = nn.Parameter(layer_scale_init_value * torch.ones(c2)) if layer_scale_init_value > 0 else None

    def forward(self, x):
        identity = self.shortcut(x)
        x = self.expand(x)
        x = self.local_mixer(x)
        x = x + self.wide_mixer(x)
        x = self.project(x)
        x = self.gate(x)
        if self.layer_scale is not None:
            x = x * self.layer_scale.view(1, -1, 1, 1)
        return self.act(identity + self.drop_path(x))


class SkyFusionStage(nn.Module):
    def __init__(self, c1: int, c2: int, depth: int, stride: int, drop_rates: Sequence[float]):
        super().__init__()
        blocks = []
        for i in range(depth):
            blocks.append(SkyFusionBlock(c1 if i == 0 else c2, c2, stride=stride if i == 0 else 1, drop_path=float(drop_rates[i]) if i < len(drop_rates) else 0.0))
        self.blocks = nn.Sequential(*blocks)

    def forward(self, x):
        return self.blocks(x)


class SkyFusionBackbone(nn.Module):
    def __init__(self, *args, c1: int = 3, c2: int = 1024, out_channels: Sequence[int] = (128, 256, 512, 1024), stage_channels: Sequence[int] = (96, 192, 384, 768), depths: Sequence[int] = (1, 2, 3, 2), drop_path_rate: float = 0.05):
        super().__init__()
        if args:
            if len(args) >= 6 or (len(args) >= 2 and isinstance(args[1], int)):
                c1, c2, out_channels, stage_channels, depths, drop_path_rate = args[:6]
            else:
                c2, out_channels, stage_channels, depths, drop_path_rate = args[:5]
        out_channels = tuple(int(x) for x in out_channels)
        stage_channels = tuple(int(x) for x in stage_channels)
        depths = tuple(int(x) for x in depths)
        drop_rates = torch.linspace(0, float(drop_path_rate), sum(depths)).tolist()
        self.c2 = c2
        self.stem = nn.Sequential(Conv(int(c1), 64, k=3, s=2), SkyFusionBlock(64, 64, stride=1, expansion=1.5))
        cursor = 0
        self.stage1 = SkyFusionStage(64, stage_channels[0], depths[0], stride=2, drop_rates=drop_rates[cursor:]); cursor += depths[0]
        self.stage2 = SkyFusionStage(stage_channels[0], stage_channels[1], depths[1], stride=2, drop_rates=drop_rates[cursor:]); cursor += depths[1]
        self.stage3 = SkyFusionStage(stage_channels[1], stage_channels[2], depths[2], stride=2, drop_rates=drop_rates[cursor:]); cursor += depths[2]
        self.stage4 = SkyFusionStage(stage_channels[2], stage_channels[3], depths[3], stride=2, drop_rates=drop_rates[cursor:])
        self.p2_proj = Conv(stage_channels[0], out_channels[0], k=1, s=1)
        self.p3_proj = Conv(stage_channels[1], out_channels[1], k=1, s=1)
        self.p4_proj = Conv(stage_channels[2], out_channels[2], k=1, s=1)
        self.p5_proj = Conv(stage_channels[3], out_channels[3], k=1, s=1)

    def forward(self, x):
        x = self.stem(x)
        p2 = self.stage1(x)
        p3 = self.stage2(p2)
        p4 = self.stage3(p3)
        p5 = self.stage4(p4)
        return [self.p2_proj(p2), self.p3_proj(p3), self.p4_proj(p4), self.p5_proj(p5)]


tasks.SkyFusionBackbone = SkyFusionBackbone
tasks.SkyFusionBlock = SkyFusionBlock
print('SkyFusion modules registered.')

In [ ]:
# 4. Write the custom YOLO26 YAML
MODEL_YAML = ROOT / 'skyfusion-yolo26s.yaml'
MODEL_YAML.write_text('''
nc: 3
scale: s
end2end: True
reg_max: 16
scales:
  n: [0.50, 0.25, 1024]
  s: [0.50, 0.50, 1024]
  m: [0.50, 1.00, 512]
  l: [1.00, 1.00, 512]
  x: [1.00, 1.50, 512]

backbone:
  - [-1, 1, SkyFusionBackbone, [1024, [128, 256, 512, 1024], [96, 192, 384, 768], [1, 2, 3, 2], 0.05]]
  - [0, 1, Index, [128, 0]]
  - [0, 1, Index, [256, 1]]
  - [0, 1, Index, [512, 2]]
  - [0, 1, Index, [1024, 3]]
  - [-1, 1, SPPF, [1024, 5, 3, True]]
  - [-1, 2, C2PSA, [1024]]

head:
  - [-1, 1, nn.Upsample, [None, 2, nearest]]
  - [[-1, 3], 1, Concat, [1]]
  - [-1, 2, C3k2, [512, True]]
  - [-1, 1, nn.Upsample, [None, 2, nearest]]
  - [[-1, 2], 1, Concat, [1]]
  - [-1, 2, C3k2, [256, True]]
  - [-1, 1, nn.Upsample, [None, 2, nearest]]
  - [[-1, 1], 1, Concat, [1]]
  - [-1, 2, C3k2, [128, True]]
  - [-1, 1, Conv, [128, 3, 2]]
  - [[-1, 12], 1, Concat, [1]]
  - [-1, 2, C3k2, [256, True]]
  - [-1, 1, Conv, [256, 3, 2]]
  - [[-1, 9], 1, Concat, [1]]
  - [-1, 2, C3k2, [512, True]]
  - [-1, 1, Conv, [512, 3, 2]]
  - [[-1, 6], 1, Concat, [1]]
  - [-1, 1, C3k2, [1024, True, 0.5, True]]
  - [[15, 18, 21, 24], 1, Detect, [nc]]
'''.strip() + '\n')

model = YOLO(str(MODEL_YAML))
model.info(detailed=False)
print(MODEL_YAML)

## Train

The assignment asks for `imgsz=640`. For a serious score, use a GPU runtime and start with 150 epochs. If Colab disconnects, set `resume=True` and point to the latest checkpoint.

In [ ]:
# 5. Train the model
EPOCHS = 150
BATCH = 16
DEVICE = 0 if torch.cuda.is_available() else 'cpu'

model = YOLO(str(MODEL_YAML))
results = model.train(
    data=str(COLAB_DATA_YAML),
    epochs=EPOCHS,
    imgsz=640,
    batch=BATCH,
    device=DEVICE,
    workers=8,
    project=str(ROOT / 'runs' / 'final_project'),
    name='skyfusion_yolo26s',
    exist_ok=True,
    pretrained=False,
    plots=True,
    cos_lr=True,
    optimizer='AdamW',
    lr0=0.0015,
    lrf=0.02,
    weight_decay=0.0005,
    warmup_epochs=3.0,
    patience=50,
    close_mosaic=20,
    mosaic=1.0,
    mixup=0.05,
    cutmix=0.05,
    hsv_h=0.015,
    hsv_s=0.60,
    hsv_v=0.35,
    degrees=7.0,
    translate=0.12,
    scale=0.55,
    shear=2.0,
    fliplr=0.5,
    flipud=0.5,
    cls_pw=0.25,
    deterministic=False,
    amp=torch.cuda.is_available(),
    cache=True,
)

BEST_PT = Path(model.trainer.save_dir) / 'weights' / 'best.pt'
print('Best checkpoint:', BEST_PT)

## Test Split Results

Run this cell after training. These are the numbers to submit with `best.pt`.

In [ ]:
# 6. Test best.pt on the SkyFusion test split
metrics = YOLO(str(BEST_PT)).val(
    data=str(COLAB_DATA_YAML),
    split='test',
    imgsz=640,
    batch=BATCH,
    device=DEVICE,
    workers=8,
    project=str(ROOT / 'runs' / 'final_project'),
    name='skyfusion_yolo26s_test',
    exist_ok=True,
    plots=True,
    save_json=True,
    conf=0.001,
    iou=0.70,
)

print(metrics.results_dict)
print(f'mAP50: {metrics.box.map50:.5f}')
print(f'mAP50-95: {metrics.box.map:.5f}')
print(f'precision: {metrics.box.mp:.5f}')
print(f'recall: {metrics.box.mr:.5f}')

In [ ]:
# 7. Optional: copy submission artifacts to Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# !mkdir -p /content/drive/MyDrive/skyfusion_submission
# !cp "{BEST_PT}" /content/drive/MyDrive/skyfusion_submission/best.pt
# !cp -r "{ROOT / 'runs' / 'final_project' / 'skyfusion_yolo26s_test'}" /content/drive/MyDrive/skyfusion_submission/test_results